In [ ]:
import pandas as pd
import os
import logging
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pyodbc

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SQLServerConnector:
    """SQL Server数据库连接和操作类"""
    
    def __init__(self, server='192.168.1.111', database='FinanceData', username='lyyz', password='AAzz0011#'):
        """
        初始化数据库连接参数
        
        Args:
            server (str): 服务器地址，默认localhost
            database (str): 数据库名称
            username (str): 用户名
            password (str): 密码
        """
        self.server = server
        self.database = database
        self.username = username
        self.password = password
        self.engine = None
        
    def get_available_drivers(self):
        """获取可用的ODBC驱动程序"""
        drivers = [x for x in pyodbc.drivers() if 'SQL Server' in x]
        logger.info(f"可用的SQL Server驱动程序: {drivers}")
        return drivers
        
    def connect(self):
        """建立数据库连接，使用SQLAlchemy"""
        # 获取可用的驱动程序
        available_drivers = self.get_available_drivers()
        
        if not available_drivers:
            logger.error("未找到SQL Server ODBC驱动程序！")
            return False
        
        # 按优先级排序驱动程序
        driver_priority = [
            "ODBC Driver 17 for SQL Server",
            "ODBC Driver 13 for SQL Server", 
            "ODBC Driver 11 for SQL Server",
            "SQL Server Native Client 11.0",
            "SQL Server"
        ]
        
        # 重新排序，优先使用更现代的驱动程序
        sorted_drivers = []
        for priority_driver in driver_priority:
            for driver in available_drivers:
                if priority_driver in driver:
                    sorted_drivers.append(driver)
                    break
        
        # 添加其他未排序的驱动程序
        for driver in available_drivers:
            if driver not in sorted_drivers:
                sorted_drivers.append(driver)
        
        logger.info(f"将按以下顺序尝试驱动程序: {sorted_drivers}")
        
        # 尝试不同的驱动程序
        for driver in sorted_drivers:
            try:
                # 构建SQLAlchemy连接字符串
                # 将驱动程序名称中的空格替换为+号，这是URL编码的要求
                driver_encoded = driver.replace(" ", "+")
                connection_string = (
                    f"mssql+pyodbc://{self.username}:{self.password}@{self.server}/"
                    f"{self.database}?driver={driver_encoded}"
                )
                
                logger.info(f"尝试使用驱动程序: {driver}")
                logger.info(f"连接字符串: {connection_string}")
                
                # 创建SQLAlchemy引擎
                self.engine = create_engine(
                    connection_string,
                    echo=False,  # 设置为True可以看到SQL语句
                    pool_pre_ping=True,  # 连接前测试连接是否有效
                    pool_recycle=3600,  # 连接回收时间（秒）
                    connect_args={
                        "timeout": 30,
                        "autocommit": True
                    }
                )
                
                # 测试连接
                with self.engine.connect() as conn:
                    conn.execute(text("SELECT 1"))
                
                logger.info(f"数据库连接成功！使用驱动程序: {driver}")
                return True
                
            except SQLAlchemyError as e:
                logger.warning(f"驱动程序 {driver} 连接失败: {e}")
                continue
        
        logger.error("所有驱动程序都无法连接")
        return False
    
    def execute_query(self, query):
        """
        执行SQL查询并返回DataFrame
        
        Args:
            query (str): SQL查询语句
            
        Returns:
            pd.DataFrame: 查询结果
        """
        try:
            if not self.engine:
                logger.error("数据库引擎未初始化")
                return None
                
            # 使用SQLAlchemy engine执行查询
            df = pd.read_sql(text(query), self.engine)
            logger.info(f"查询执行成功，返回 {len(df)} 行数据")
            return df
            
        except SQLAlchemyError as e:
            logger.error(f"SQLAlchemy查询执行失败: {e}")
            return None
        except Exception as e:
            logger.error(f"查询执行失败: {e}")
            return None
    
    def close(self):
        """关闭数据库连接"""
        if self.engine:
            self.engine.dispose()
            logger.info("数据库连接已关闭")



In [ ]:

# 使用实际存在的表进行查询
queries = {
    'shangzheng50_PB':
    """
                With RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                    )
                SELECT 
                    p.Date AS [datetime],         
                    p.Code AS code,
                CASE 
                        WHEN r.Title IS NOT NULL THEN '-99998'
                        WHEN p.Date = m.S_CON_OUTDATE THEN '-99999'
                        WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN p.PB
                        ELSE NULL 
                    END AS pb

                FROM FinanceData..AShare_PriceDaily AS p
                INNER JOIN FinanceData..AINDEXMEMBERS AS m  
                        ON p.Code = m.Code  
                        AND m.S_IRDCODE = '000016.SH' 
                LEFT JOIN RiskAnnouncements AS r
                        ON p.Date = r.Date 
                        AND p.Code = r.Code
                order by p.Date,p.Code
    
    """
    ,'hushen300_PB':
    """
                With RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                    )
                SELECT 
                    p.Date AS [datetime],         
                    p.Code AS code,
                CASE 
                        WHEN r.Title IS NOT NULL THEN '-99998'
                        WHEN p.Date = m.S_CON_OUTDATE THEN '-99999'
                        WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN p.PB
                        ELSE NULL 
                    END AS pb

                FROM FinanceData..AShare_PriceDaily AS p
                INNER JOIN FinanceData..AINDEXMEMBERS AS m  
                        ON p.Code = m.Code  
                        AND m.S_IRDCODE = '000300.SH' 
                LEFT JOIN RiskAnnouncements AS r
                        ON p.Date = r.Date 
                        AND p.Code = r.Code
                order by p.Date,p.Code
    
    """
    ,'zhongzheng500_PB':
    """
                With RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                    )
                SELECT 
                    p.Date AS [datetime],         
                    p.Code AS code,
                CASE 
                        WHEN r.Title IS NOT NULL THEN '-99998'
                        WHEN p.Date = m.S_CON_OUTDATE THEN '-99999'
                        WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN p.PB
                        ELSE NULL 
                    END AS pb

                FROM FinanceData..AShare_PriceDaily AS p
                INNER JOIN FinanceData..AINDEXMEMBERS AS m  
                        ON p.Code = m.Code  
                        AND m.S_IRDCODE = '000905.SH' 
                LEFT JOIN RiskAnnouncements AS r
                        ON p.Date = r.Date 
                        AND p.Code = r.Code
                order by p.Date,p.Code
    
    """
    ,'zhongzheng1000_PB':
    """
                With RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                    )
                SELECT 
                    p.Date AS [datetime],         
                    p.Code AS code,
                CASE 
                        WHEN r.Title IS NOT NULL THEN '-99998'
                        WHEN p.Date = m.S_CON_OUTDATE THEN '-99999'
                        WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN p.PB
                        ELSE NULL 
                    END AS pb

                FROM FinanceData..AShare_PriceDaily AS p
                INNER JOIN FinanceData..AINDEXMEMBERS AS m  
                        ON p.Code = m.Code  
                        AND m.S_IRDCODE = '000852.SH' 
                LEFT JOIN RiskAnnouncements AS r
                        ON p.Date = r.Date 
                        AND p.Code = r.Code
                order by p.Date,p.Code
    
    """
        ,'allAshare_PB':
    """
                    With delistmember as(
                        SELECT code,IPOdate,DelistDate
                        FROM (
                            SELECT *,
                                ROW_NUMBER() OVER (PARTITION BY code ORDER BY date) as rn
                            FROM FinanceData..AShare_StockInfo
                        ) t
                        WHERE rn = 1
                    )
                    ,RiskAnnouncements AS (
                        SELECT 
                            Date, 
                            Code, 
                            Title
                        FROM FinanceData.dbo.Web_AnnounceIndex
                        WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                        )
                    SELECT 
                        p.Date AS [datetime],         
                        p.Code AS code,
                    CASE 
                            WHEN r.Title IS NOT NULL THEN '-99998'
                            WHEN p.Date = m.DelistDate THEN '-99999'
                            WHEN p.Date BETWEEN m.IPODate AND COALESCE(CASE WHEN m.delistdate = '0' THEN NULL ELSE m.delistdate END, '2099-01-01') THEN p.PB
                            ELSE NULL 
                        END AS pb

                    FROM FinanceData..AShare_PriceDaily AS p
                    INNER JOIN delistmember AS m  
                            ON p.Code = m.Code  
                        
                    LEFT JOIN RiskAnnouncements AS r
                            ON p.Date = r.Date 
                            AND p.Code = r.Code
                    order by p.Date,p.Code

    
    """
    ,'shangzheng50_change20':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000016.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'hushen300_change20':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000300.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'zhongzheng500_change20':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000905.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'zhongzheng1000_change20':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000852.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'allAshare_change20':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 20) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'shangzheng50_change60':
    """
            
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000016.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
        ,'hushen300_change60':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000300.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'zhongzheng500_change60':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000905.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'zhongzheng1000_change60':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000852.SH'
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'allAshare_change60':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                
                    WHEN w.Title IS NOT NULL THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    WHEN p.Date BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') 
                        THEN (p.AdjPrice - LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date)) 
                        / LAG(p.AdjPrice, 60) OVER (PARTITION BY p.Code ORDER BY p.Date) * 100
                    ELSE NULL 
                END AS change_percent_20d
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime];
    """
    ,'allAshareStock_data':
    """
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                p.Name AS name,
                p.TradeStatus as tradestatus,
                p.AdjPrice * p.OpenPrice / p.ClosePrice AS [open],  
                p.AdjPrice * p.HighPrice / p.ClosePrice AS [high],
                p.AdjPrice * p.LowPrice / p.ClosePrice AS [low],
                p.AdjPrice AS [close],
                p.Volume AS [volume], 
                (p.AdjPrice-(p.AdjPrice * p.OpenPrice / p.ClosePrice))/p.AdjPrice as [change],
                p.PB              
            FROM FinanceData..AShare_PriceDaily as p
            where p.Date>'2015-01-01'
    """
    ,'shangzheng50_high250':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                    WHEN w.Title IS NOT NULL  THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    ELSE (p.AdjPrice / NULLIF(MAX(p.AdjPrice) OVER (
                        PARTITION BY p.Code 
                        ORDER BY p.Date 
                        ROWS BETWEEN 250 PRECEDING AND 1 PRECEDING
                    ), 0)) - 1
                END AS high250
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000016.SH'--指数参数
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime]
    """
    ,'hushen300_high250':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                    WHEN w.Title IS NOT NULL  THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    ELSE (p.AdjPrice / NULLIF(MAX(p.AdjPrice) OVER (
                        PARTITION BY p.Code 
                        ORDER BY p.Date 
                        ROWS BETWEEN 250 PRECEDING AND 1 PRECEDING
                    ), 0)) - 1
                END AS high250
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000300.SH'--指数参数
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime]
    """
    ,'zhongzheng500_high250':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                    WHEN w.Title IS NOT NULL  THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    ELSE (p.AdjPrice / NULLIF(MAX(p.AdjPrice) OVER (
                        PARTITION BY p.Code 
                        ORDER BY p.Date 
                        ROWS BETWEEN 250 PRECEDING AND 1 PRECEDING
                    ), 0)) - 1
                END AS high250
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000905.SH'--指数参数
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime]
    """
    ,'zhongzheng1000_high250':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                    WHEN w.Title IS NOT NULL  THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    ELSE (p.AdjPrice / NULLIF(MAX(p.AdjPrice) OVER (
                        PARTITION BY p.Code 
                        ORDER BY p.Date 
                        ROWS BETWEEN 250 PRECEDING AND 1 PRECEDING
                    ), 0)) - 1
                END AS high250
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
                AND m.S_IRDCODE = '000852.SH'--指数参数
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime]
    """
    ,'allAshare_high250':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            SELECT 
                p.Date AS [datetime],         
                p.Code AS code,
                CASE 
                    WHEN w.Title IS NOT NULL  THEN -99998
                    WHEN p.Date = m.S_CON_OUTDATE THEN -99999
                    ELSE (p.AdjPrice / NULLIF(MAX(p.AdjPrice) OVER (
                        PARTITION BY p.Code 
                        ORDER BY p.Date 
                        ROWS BETWEEN 250 PRECEDING AND 1 PRECEDING
                    ), 0)) - 1
                END AS high250
            FROM FinanceData..AShare_PriceDaily AS p
            INNER JOIN FinanceData..AINDEXMEMBERS AS m 
                ON p.Code = m.Code  
            LEFT JOIN RiskAnnouncements AS w
                ON p.Date = w.Date 
                AND p.Code = w.Code
            ORDER BY [datetime]
    """
    ,'shangzheng50_abnormal_gross_margin':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            ,CombinedData AS (
                SELECT 
                    b.code,
                    b.REPORT_PERIOD,
                    b.ANN_DT,
                    b.TOT_ASSETS,
                    i.TOT_OPER_REV,
                    i.TOT_OPER_COST,
                    c.CASH_RECP_SG_AND_RS
                FROM FinanceData..ASHAREBALANCESHEET b
                LEFT JOIN FinanceData..ASHAREINCOME i 
                    ON b.code = i.code 
                    AND b.REPORT_PERIOD = i.REPORT_PERIOD
                LEFT JOIN FinanceData..ASHARECASHFLOW c 
                    ON b.code = c.code 
                    AND b.REPORT_PERIOD = c.REPORT_PERIOD
                WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000' and c.STATEMENT_TYPE ='408001000'
            )
            ,FactorCalculate_preparation AS (
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS,
                    TOT_OPER_REV,
                    TOT_OPER_COST,
                    CASH_RECP_SG_AND_RS,
                    -- 计算毛利润
                    TOT_OPER_REV - TOT_OPER_COST AS gross_profit,
                    -- 使用LAG获取去年同期数据（假设数据按季度连续）
                    LAG(TOT_OPER_REV, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_REV_LY,
                    LAG(TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_COST_LY,
                    LAG(CASH_RECP_SG_AND_RS, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS CASH_RECP_SG_AND_RS_LY,
                    -- 计算去年同期毛利润
                    LAG(TOT_OPER_REV - TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS gross_profit_LY
                FROM CombinedData
            )
            ,cal_normal_growth_multiplier AS (
                -- 第一步：选择必要的字段，并进行初步计算
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS AS 当季末总资产,
                    gross_profit AS 当期毛利润,
                    gross_profit_LY AS 去年同期毛利润,
                    CASH_RECP_SG_AND_RS AS 当期销售现金,
                    CASH_RECP_SG_AND_RS_LY AS 去年同期销售现金,
                
                    CASE 
                        WHEN CASH_RECP_SG_AND_RS_LY IS NOT NULL 
                            AND CASH_RECP_SG_AND_RS_LY <> 0
                        THEN CASH_RECP_SG_AND_RS * 1.0 / CASH_RECP_SG_AND_RS_LY
                        ELSE NULL 
                    END AS 正常增长乘数
                FROM FactorCalculate_preparation
            )
            ,AbnormalMarginData AS (
            SELECT 
                code,
                ANN_DT as [datetime],
                -- 计算异常毛利率，此时逻辑得到简化
                CASE 
                    WHEN 当季末总资产 IS NOT NULL 
                            AND 当季末总资产 <> 0 
                            AND 当期毛利润 IS NOT NULL
                            AND 去年同期毛利润 IS NOT NULL
                    THEN (当期毛利润 - 去年同期毛利润 * COALESCE(正常增长乘数, 1)) 
                            * 1.0 / 当季末总资产
                    ELSE NULL 
                END AS abnormal_gross_margin
            FROM cal_normal_growth_multiplier

            )
            SELECT distinct
                a.code,
                datetime,
                CASE 
                    WHEN r.Title IS NOT NULL THEN -99998
                    WHEN a.datetime = m.S_CON_OUTDATE THEN -99999
                    WHEN a.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  a.abnormal_gross_margin
                    ELSE null
                END AS abnormal_gross_margin
            FROM AbnormalMarginData a

            left join RiskAnnouncements r
            on a.datetime =r.Date
            and a.CODE =r.Code

            JOIN FinanceData..AINDEXMEMBERS AS m 
                ON a.Code = m.Code  
                AND m.S_IRDCODE = '000016.SH'
            order by a.CODE , a.datetime
    """
    ,'hushen300_abnormal_gross_margin':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            ,CombinedData AS (
                SELECT 
                    b.code,
                    b.REPORT_PERIOD,
                    b.ANN_DT,
                    b.TOT_ASSETS,
                    i.TOT_OPER_REV,
                    i.TOT_OPER_COST,
                    c.CASH_RECP_SG_AND_RS
                FROM FinanceData..ASHAREBALANCESHEET b
                LEFT JOIN FinanceData..ASHAREINCOME i 
                    ON b.code = i.code 
                    AND b.REPORT_PERIOD = i.REPORT_PERIOD
                LEFT JOIN FinanceData..ASHARECASHFLOW c 
                    ON b.code = c.code 
                    AND b.REPORT_PERIOD = c.REPORT_PERIOD
                WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000' and c.STATEMENT_TYPE ='408001000'
            )
            ,FactorCalculate_preparation AS (
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS,
                    TOT_OPER_REV,
                    TOT_OPER_COST,
                    CASH_RECP_SG_AND_RS,
                    -- 计算毛利润
                    TOT_OPER_REV - TOT_OPER_COST AS gross_profit,
                    -- 使用LAG获取去年同期数据（假设数据按季度连续）
                    LAG(TOT_OPER_REV, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_REV_LY,
                    LAG(TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_COST_LY,
                    LAG(CASH_RECP_SG_AND_RS, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS CASH_RECP_SG_AND_RS_LY,
                    -- 计算去年同期毛利润
                    LAG(TOT_OPER_REV - TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS gross_profit_LY
                FROM CombinedData
            )
            ,cal_normal_growth_multiplier AS (
                -- 第一步：选择必要的字段，并进行初步计算
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS AS 当季末总资产,
                    gross_profit AS 当期毛利润,
                    gross_profit_LY AS 去年同期毛利润,
                    CASH_RECP_SG_AND_RS AS 当期销售现金,
                    CASH_RECP_SG_AND_RS_LY AS 去年同期销售现金,
                
                    CASE 
                        WHEN CASH_RECP_SG_AND_RS_LY IS NOT NULL 
                            AND CASH_RECP_SG_AND_RS_LY <> 0
                        THEN CASH_RECP_SG_AND_RS * 1.0 / CASH_RECP_SG_AND_RS_LY
                        ELSE NULL 
                    END AS 正常增长乘数
                FROM FactorCalculate_preparation
            )
            ,AbnormalMarginData AS (
            SELECT 
                code,
                ANN_DT as [datetime],
                -- 计算异常毛利率，此时逻辑得到简化
                CASE 
                    WHEN 当季末总资产 IS NOT NULL 
                            AND 当季末总资产 <> 0 
                            AND 当期毛利润 IS NOT NULL
                            AND 去年同期毛利润 IS NOT NULL
                    THEN (当期毛利润 - 去年同期毛利润 * COALESCE(正常增长乘数, 1)) 
                            * 1.0 / 当季末总资产
                    ELSE NULL 
                END AS abnormal_gross_margin
            FROM cal_normal_growth_multiplier

            )
            SELECT distinct
                a.code,
                datetime,
                CASE 
                    WHEN r.Title IS NOT NULL THEN -99998
                    WHEN a.datetime = m.S_CON_OUTDATE THEN -99999
                    WHEN a.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  a.abnormal_gross_margin
                    ELSE null
                END AS abnormal_gross_margin
            FROM AbnormalMarginData a

            left join RiskAnnouncements r
            on a.datetime =r.Date
            and a.CODE =r.Code

            JOIN FinanceData..AINDEXMEMBERS AS m 
                ON a.Code = m.Code  
                AND m.S_IRDCODE = '000300.SH'
            order by a.CODE , a.datetime
    """
    ,'zhongzheng500_abnormal_gross_margin':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            ,CombinedData AS (
                SELECT 
                    b.code,
                    b.REPORT_PERIOD,
                    b.ANN_DT,
                    b.TOT_ASSETS,
                    i.TOT_OPER_REV,
                    i.TOT_OPER_COST,
                    c.CASH_RECP_SG_AND_RS
                FROM FinanceData..ASHAREBALANCESHEET b
                LEFT JOIN FinanceData..ASHAREINCOME i 
                    ON b.code = i.code 
                    AND b.REPORT_PERIOD = i.REPORT_PERIOD
                LEFT JOIN FinanceData..ASHARECASHFLOW c 
                    ON b.code = c.code 
                    AND b.REPORT_PERIOD = c.REPORT_PERIOD
                WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000' and c.STATEMENT_TYPE ='408001000'
            )
            ,FactorCalculate_preparation AS (
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS,
                    TOT_OPER_REV,
                    TOT_OPER_COST,
                    CASH_RECP_SG_AND_RS,
                    -- 计算毛利润
                    TOT_OPER_REV - TOT_OPER_COST AS gross_profit,
                    -- 使用LAG获取去年同期数据（假设数据按季度连续）
                    LAG(TOT_OPER_REV, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_REV_LY,
                    LAG(TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_COST_LY,
                    LAG(CASH_RECP_SG_AND_RS, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS CASH_RECP_SG_AND_RS_LY,
                    -- 计算去年同期毛利润
                    LAG(TOT_OPER_REV - TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS gross_profit_LY
                FROM CombinedData
            )
            ,cal_normal_growth_multiplier AS (
                -- 第一步：选择必要的字段，并进行初步计算
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS AS 当季末总资产,
                    gross_profit AS 当期毛利润,
                    gross_profit_LY AS 去年同期毛利润,
                    CASH_RECP_SG_AND_RS AS 当期销售现金,
                    CASH_RECP_SG_AND_RS_LY AS 去年同期销售现金,
                
                    CASE 
                        WHEN CASH_RECP_SG_AND_RS_LY IS NOT NULL 
                            AND CASH_RECP_SG_AND_RS_LY <> 0
                        THEN CASH_RECP_SG_AND_RS * 1.0 / CASH_RECP_SG_AND_RS_LY
                        ELSE NULL 
                    END AS 正常增长乘数
                FROM FactorCalculate_preparation
            )
            ,AbnormalMarginData AS (
            SELECT 
                code,
                ANN_DT as [datetime],
                -- 计算异常毛利率，此时逻辑得到简化
                CASE 
                    WHEN 当季末总资产 IS NOT NULL 
                            AND 当季末总资产 <> 0 
                            AND 当期毛利润 IS NOT NULL
                            AND 去年同期毛利润 IS NOT NULL
                    THEN (当期毛利润 - 去年同期毛利润 * COALESCE(正常增长乘数, 1)) 
                            * 1.0 / 当季末总资产
                    ELSE NULL 
                END AS abnormal_gross_margin
            FROM cal_normal_growth_multiplier

            )
            SELECT distinct
                a.code,
                datetime,
                CASE 
                    WHEN r.Title IS NOT NULL THEN -99998
                    WHEN a.datetime = m.S_CON_OUTDATE THEN -99999
                    WHEN a.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  a.abnormal_gross_margin
                    ELSE null
                END AS abnormal_gross_margin
            FROM AbnormalMarginData a

            left join RiskAnnouncements r
            on a.datetime =r.Date
            and a.CODE =r.Code

            JOIN FinanceData..AINDEXMEMBERS AS m 
                ON a.Code = m.Code  
                AND m.S_IRDCODE = '000905.SH'
            order by a.CODE , a.datetime
    """
    ,'zhongzheng1000_abnormal_gross_margin':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            ,CombinedData AS (
                SELECT 
                    b.code,
                    b.REPORT_PERIOD,
                    b.ANN_DT,
                    b.TOT_ASSETS,
                    i.TOT_OPER_REV,
                    i.TOT_OPER_COST,
                    c.CASH_RECP_SG_AND_RS
                FROM FinanceData..ASHAREBALANCESHEET b
                LEFT JOIN FinanceData..ASHAREINCOME i 
                    ON b.code = i.code 
                    AND b.REPORT_PERIOD = i.REPORT_PERIOD
                LEFT JOIN FinanceData..ASHARECASHFLOW c 
                    ON b.code = c.code 
                    AND b.REPORT_PERIOD = c.REPORT_PERIOD
                WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000' and c.STATEMENT_TYPE ='408001000'
            )
            ,FactorCalculate_preparation AS (
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS,
                    TOT_OPER_REV,
                    TOT_OPER_COST,
                    CASH_RECP_SG_AND_RS,
                    -- 计算毛利润
                    TOT_OPER_REV - TOT_OPER_COST AS gross_profit,
                    -- 使用LAG获取去年同期数据（假设数据按季度连续）
                    LAG(TOT_OPER_REV, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_REV_LY,
                    LAG(TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_COST_LY,
                    LAG(CASH_RECP_SG_AND_RS, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS CASH_RECP_SG_AND_RS_LY,
                    -- 计算去年同期毛利润
                    LAG(TOT_OPER_REV - TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS gross_profit_LY
                FROM CombinedData
            )
            ,cal_normal_growth_multiplier AS (
                -- 第一步：选择必要的字段，并进行初步计算
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS AS 当季末总资产,
                    gross_profit AS 当期毛利润,
                    gross_profit_LY AS 去年同期毛利润,
                    CASH_RECP_SG_AND_RS AS 当期销售现金,
                    CASH_RECP_SG_AND_RS_LY AS 去年同期销售现金,
                
                    CASE 
                        WHEN CASH_RECP_SG_AND_RS_LY IS NOT NULL 
                            AND CASH_RECP_SG_AND_RS_LY <> 0
                        THEN CASH_RECP_SG_AND_RS * 1.0 / CASH_RECP_SG_AND_RS_LY
                        ELSE NULL 
                    END AS 正常增长乘数
                FROM FactorCalculate_preparation
            )
            ,AbnormalMarginData AS (
            SELECT 
                code,
                ANN_DT as [datetime],
                -- 计算异常毛利率，此时逻辑得到简化
                CASE 
                    WHEN 当季末总资产 IS NOT NULL 
                            AND 当季末总资产 <> 0 
                            AND 当期毛利润 IS NOT NULL
                            AND 去年同期毛利润 IS NOT NULL
                    THEN (当期毛利润 - 去年同期毛利润 * COALESCE(正常增长乘数, 1)) 
                            * 1.0 / 当季末总资产
                    ELSE NULL 
                END AS abnormal_gross_margin
            FROM cal_normal_growth_multiplier

            )
            SELECT distinct
                a.code,
                datetime,
                CASE 
                    WHEN r.Title IS NOT NULL THEN -99998
                    WHEN a.datetime = m.S_CON_OUTDATE THEN -99999
                    WHEN a.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  a.abnormal_gross_margin
                    ELSE null
                END AS abnormal_gross_margin
            FROM AbnormalMarginData a

            left join RiskAnnouncements r
            on a.datetime =r.Date
            and a.CODE =r.Code

            JOIN FinanceData..AINDEXMEMBERS AS m 
                ON a.Code = m.Code  
                AND m.S_IRDCODE = '000852.SH'
            order by a.CODE , a.datetime
    """
    ,'abnormal_gross_margin':
    """
            WITH RiskAnnouncements AS (
                SELECT 
                    Date, 
                    Code, 
                    Title
                FROM FinanceData.dbo.Web_AnnounceIndex
                WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
            )
            ,CombinedData AS (
                SELECT 
                    b.code,
                    b.REPORT_PERIOD,
                    b.ANN_DT,
                    b.TOT_ASSETS,
                    i.TOT_OPER_REV,
                    i.TOT_OPER_COST,
                    c.CASH_RECP_SG_AND_RS
                FROM FinanceData..ASHAREBALANCESHEET b
                LEFT JOIN FinanceData..ASHAREINCOME i 
                    ON b.code = i.code 
                    AND b.REPORT_PERIOD = i.REPORT_PERIOD
                LEFT JOIN FinanceData..ASHARECASHFLOW c 
                    ON b.code = c.code 
                    AND b.REPORT_PERIOD = c.REPORT_PERIOD
                WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000' and c.STATEMENT_TYPE ='408001000'
            )
            ,FactorCalculate_preparation AS (
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS,
                    TOT_OPER_REV,
                    TOT_OPER_COST,
                    CASH_RECP_SG_AND_RS,
                    -- 计算毛利润
                    TOT_OPER_REV - TOT_OPER_COST AS gross_profit,
                    -- 使用LAG获取去年同期数据（假设数据按季度连续）
                    LAG(TOT_OPER_REV, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_REV_LY,
                    LAG(TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS TOT_OPER_COST_LY,
                    LAG(CASH_RECP_SG_AND_RS, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS CASH_RECP_SG_AND_RS_LY,
                    -- 计算去年同期毛利润
                    LAG(TOT_OPER_REV - TOT_OPER_COST, 4) OVER (PARTITION BY code ORDER BY REPORT_PERIOD) AS gross_profit_LY
                FROM CombinedData
            )
            ,cal_normal_growth_multiplier AS (
                -- 第一步：选择必要的字段，并进行初步计算
                SELECT 
                    code,
                    REPORT_PERIOD,
                    ANN_DT,
                    TOT_ASSETS AS 当季末总资产,
                    gross_profit AS 当期毛利润,
                    gross_profit_LY AS 去年同期毛利润,
                    CASH_RECP_SG_AND_RS AS 当期销售现金,
                    CASH_RECP_SG_AND_RS_LY AS 去年同期销售现金,
                                
                    CASE 
                        WHEN CASH_RECP_SG_AND_RS_LY IS NOT NULL 
                            AND CASH_RECP_SG_AND_RS_LY <> 0
                        THEN CASH_RECP_SG_AND_RS * 1.0 / CASH_RECP_SG_AND_RS_LY
                        ELSE NULL 
                    END AS 正常增长乘数
                FROM FactorCalculate_preparation
            )
            SELECT 
                code,
                ANN_DT ,
                -- 计算异常毛利率，此时逻辑得到简化
                CASE 
                    WHEN 当季末总资产 IS NOT NULL 
                            AND 当季末总资产 <> 0 
                            AND 当期毛利润 IS NOT NULL
                            AND 去年同期毛利润 IS NOT NULL
                    THEN (当期毛利润 - 去年同期毛利润 * COALESCE(正常增长乘数, 1)) 
                            * 1.0 / 当季末总资产
                    ELSE NULL 
                END AS abnormal_gross_margin
            FROM cal_normal_growth_multiplier

    """
    ,'shangzheng50_CFO':
    """
                WITH RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                )
                ,CombinedData AS (
                    SELECT 
                        b.code,
                        b.REPORT_PERIOD,
                        b.ANN_DT as [datetime],
                        b.TOT_ASSETS,

                        c.NET_CASH_FLOWS_OPER_ACT
                    FROM FinanceData..ASHAREBALANCESHEET b
                    LEFT JOIN FinanceData..ASHARECASHFLOW c 
                        ON b.code = c.code 
                        AND b.REPORT_PERIOD = c.REPORT_PERIOD
                    WHERE b.STATEMENT_TYPE = '408001000'  and c.STATEMENT_TYPE ='408001000'
                )
                SELECT distinct
                    c.code,
                    datetime,
                    CASE 
                        WHEN r.Title IS NOT NULL THEN -99998
                        WHEN c.datetime = m.S_CON_OUTDATE THEN -99999
                        WHEN c.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  c.NET_CASH_FLOWS_OPER_ACT/c.TOT_ASSETS
                        ELSE null
                    END AS CFO
                FROM CombinedData c

                left join RiskAnnouncements r
                on c.datetime =r.Date
                and c.CODE =r.Code

                JOIN FinanceData..AINDEXMEMBERS AS m 
                    ON c.Code = m.Code  
                    AND m.S_IRDCODE = '000016.SH'
                order by c.CODE , c.datetime
    """
    ,'hushen300_CFO':
    """
                WITH RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                )
                ,CombinedData AS (
                    SELECT 
                        b.code,
                        b.REPORT_PERIOD,
                        b.ANN_DT as [datetime],
                        b.TOT_ASSETS,

                        c.NET_CASH_FLOWS_OPER_ACT
                    FROM FinanceData..ASHAREBALANCESHEET b
                    LEFT JOIN FinanceData..ASHARECASHFLOW c 
                        ON b.code = c.code 
                        AND b.REPORT_PERIOD = c.REPORT_PERIOD
                    WHERE b.STATEMENT_TYPE = '408001000'  and c.STATEMENT_TYPE ='408001000'
                )
                SELECT distinct
                    c.code,
                    datetime,
                    CASE 
                        WHEN r.Title IS NOT NULL THEN -99998
                        WHEN c.datetime = m.S_CON_OUTDATE THEN -99999
                        WHEN c.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  c.NET_CASH_FLOWS_OPER_ACT/c.TOT_ASSETS
                        ELSE null
                    END AS CFO
                FROM CombinedData c

                left join RiskAnnouncements r
                on c.datetime =r.Date
                and c.CODE =r.Code

                JOIN FinanceData..AINDEXMEMBERS AS m 
                    ON c.Code = m.Code  
                    AND m.S_IRDCODE = '000300.SH'
                order by c.CODE , c.datetime
    """
    ,'zhongzheng500_CFO':
    """
                WITH RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                )
                ,CombinedData AS (
                    SELECT 
                        b.code,
                        b.REPORT_PERIOD,
                        b.ANN_DT as [datetime],
                        b.TOT_ASSETS,

                        c.NET_CASH_FLOWS_OPER_ACT
                    FROM FinanceData..ASHAREBALANCESHEET b
                    LEFT JOIN FinanceData..ASHARECASHFLOW c 
                        ON b.code = c.code 
                        AND b.REPORT_PERIOD = c.REPORT_PERIOD
                    WHERE b.STATEMENT_TYPE = '408001000'  and c.STATEMENT_TYPE ='408001000'
                )
                SELECT distinct
                    c.code,
                    datetime,
                    CASE 
                        WHEN r.Title IS NOT NULL THEN -99998
                        WHEN c.datetime = m.S_CON_OUTDATE THEN -99999
                        WHEN c.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  c.NET_CASH_FLOWS_OPER_ACT/c.TOT_ASSETS
                        ELSE null
                    END AS CFO
                FROM CombinedData c

                left join RiskAnnouncements r
                on c.datetime =r.Date
                and c.CODE =r.Code

                JOIN FinanceData..AINDEXMEMBERS AS m 
                    ON c.Code = m.Code  
                    AND m.S_IRDCODE = '000905.SH'
                order by c.CODE , c.datetime
    """
    ,'zhongzheng1000_CFO':
    """
                WITH RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                )
                ,CombinedData AS (
                    SELECT 
                        b.code,
                        b.REPORT_PERIOD,
                        b.ANN_DT as [datetime],
                        b.TOT_ASSETS,

                        c.NET_CASH_FLOWS_OPER_ACT
                    FROM FinanceData..ASHAREBALANCESHEET b
                    LEFT JOIN FinanceData..ASHARECASHFLOW c 
                        ON b.code = c.code 
                        AND b.REPORT_PERIOD = c.REPORT_PERIOD
                    WHERE b.STATEMENT_TYPE = '408001000'  and c.STATEMENT_TYPE ='408001000'
                )
                SELECT distinct
                    c.code,
                    datetime,
                    CASE 
                        WHEN r.Title IS NOT NULL THEN -99998
                        WHEN c.datetime = m.S_CON_OUTDATE THEN -99999
                        WHEN c.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  c.NET_CASH_FLOWS_OPER_ACT/c.TOT_ASSETS
                        ELSE null
                    END AS CFO
                FROM CombinedData c

                left join RiskAnnouncements r
                on c.datetime =r.Date
                and c.CODE =r.Code

                JOIN FinanceData..AINDEXMEMBERS AS m 
                    ON c.Code = m.Code  
                    AND m.S_IRDCODE = '000852.SH'
                order by c.CODE , c.datetime
    """
    ,'allAshare_CFO':
    """
                WITH RiskAnnouncements AS (
                    SELECT 
                        Date, 
                        Code, 
                        Title
                    FROM FinanceData.dbo.Web_AnnounceIndex
                    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
                )
                ,CombinedData AS (
                    SELECT 
                        b.code,
                        b.REPORT_PERIOD,
                        b.ANN_DT as [datetime],
                        b.TOT_ASSETS,

                        c.NET_CASH_FLOWS_OPER_ACT
                    FROM FinanceData..ASHAREBALANCESHEET b
                    LEFT JOIN FinanceData..ASHARECASHFLOW c 
                        ON b.code = c.code 
                        AND b.REPORT_PERIOD = c.REPORT_PERIOD
                    WHERE b.STATEMENT_TYPE = '408001000'  and c.STATEMENT_TYPE ='408001000'
                )
                SELECT distinct
                    c.code,
                    datetime,
                    CASE 
                        WHEN r.Title IS NOT NULL THEN -99998
                        WHEN c.datetime = m.S_CON_OUTDATE THEN -99999
                        WHEN c.datetime BETWEEN m.S_CON_INDATE AND COALESCE(m.S_CON_OUTDATE, '2099-01-01') THEN  c.NET_CASH_FLOWS_OPER_ACT/c.TOT_ASSETS
                        ELSE null
                    END AS CFO
                FROM CombinedData c

                left join RiskAnnouncements r
                on c.datetime =r.Date
                and c.CODE =r.Code

                JOIN FinanceData..AINDEXMEMBERS AS m 
                    ON c.Code = m.Code  
                    
                order by c.CODE , c.datetime
    """
    ,'RiskAnnouncements':
    """
    SELECT
        Date as datetime,
        Code as code,
        Title as title
    FROM FinanceData.dbo.Web_AnnounceIndex
    WHERE CONTAINS(Title, '"风险警示" OR "立案调查" OR "合并" OR "暂停上市"')
    """
    ,'Delta_ROA':
    """
    SELECT
        b.code,
        b.REPORT_PERIOD,
        b.ANN_DT,
        b.TOT_ASSETS,
        i.NET_PROFIT_INCL_MIN_INT_INC,
        -- 使用LAG获取去年同期数据（假设数据按季度连续）
        LAG(b.TOT_ASSETS, 4) OVER (PARTITION BY b.code ORDER BY b.REPORT_PERIOD) AS TOT_ASSETS_LY,
        LAG(i.NET_PROFIT_INCL_MIN_INT_INC, 4) OVER (PARTITION BY b.code ORDER BY b.REPORT_PERIOD) AS NET_PROFIT_INCL_MIN_INT_INC_LY

    FROM FinanceData..ASHAREBALANCESHEET b
    JOIN FinanceData..ASHAREINCOME i ON b.code = i.code AND b.REPORT_PERIOD = i.REPORT_PERIOD
    WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000'
    """
    ,'Delta_ROE':
    """
    SELECT
        b.code,
        b.REPORT_PERIOD,
        b.ANN_DT,
        b.TOT_ASSETS,
        b.TOT_LIAB,
        TOT_ASSETS-TOT_LIAB as TOT_NET_ASSETS,
        i.NET_PROFIT_INCL_MIN_INT_INC,
        -- 使用LAG获取去年同期数据（假设数据按季度连续）
        LAG(b.TOT_ASSETS, 4) OVER (PARTITION BY b.code ORDER BY b.REPORT_PERIOD) AS TOT_ASSETS_LY,
        LAG(b.TOT_LIAB, 4) OVER (PARTITION BY b.code ORDER BY b.REPORT_PERIOD) AS TOT_LIAB_LY,
        LAG(i.NET_PROFIT_INCL_MIN_INT_INC, 4) OVER (PARTITION BY b.code ORDER BY b.REPORT_PERIOD) AS NET_PROFIT_INCL_MIN_INT_INC_LY

    FROM FinanceData..ASHAREBALANCESHEET b
    JOIN FinanceData..ASHAREINCOME i ON b.code = i.code AND b.REPORT_PERIOD = i.REPORT_PERIOD
    WHERE b.STATEMENT_TYPE = '408001000' and i.STATEMENT_TYPE='408001000'
    """
    ,'AINDEXMEMBERS':
    """
        select
            CODE as code,
            S_IRDCODE,
            S_CON_INDATE,
            S_CON_OUTDATE
        from FinanceData..AINDEXMEMBERS
    """
    ,"CFO":
    """
    SELECT
        b.code
        ,b.REPORT_PERIOD
        ,b.ANN_DT
        ,b.TOT_ASSETS
        ,c.NET_CASH_FLOWS_OPER_ACT
            ,c.NET_CASH_FLOWS_OPER_ACT/b.TOT_ASSETS as CFO

    FROM FinanceData..ASHAREBALANCESHEET b
    LEFT JOIN FinanceData..ASHARECASHFLOW c ON b.code = c.code AND b.REPORT_PERIOD = c.REPORT_PERIOD
    WHERE b.STATEMENT_TYPE = '408001000' and c.STATEMENT_TYPE='408001000' and b.TOT_ASSETS !=0
    """
    ,'ashare_holdreward':
    """
        select * 
        FROM OPENQUERY(WIND,  'select S_INFO_WINDCODE, ANN_DATE, END_DATE, CRNY_CODE, S_INFO_MANAGER_NAME, S_INFO_MANAGER_POST, S_MANAGER_RETURN, S_MANAGER_QUANTITY, MANID, S_MANAGER_RETURN_OTHER, OPDATE, OPMODE from wind.AShareManagementHoldReward
        ')
    """
    ,'marketvalue':
    """
        select 
        FROM OPENQUERY(WIND,  'select S_INFO_WINDCODE, ANN_DATE, END_DATE, CRNY_CODE, S_INFO_MANAGER_NAME, S_INFO_MANAGER_POST, S_MANAGER_RETURN, S_MANAGER_QUANTITY, MANID, S_MANAGER_RETURN_OTHER, OPDATE, OPMODE from wind.AShareManagementHoldReward
        ')
    """
    ,'ashare_pricedaily':
    """
        SELECT 
            [Code]
            ,[Date]
            ,[Name]
            ,[TradeStatus]
            ,[PreClose]
            ,[OpenPrice]
            ,[HighPrice]
            ,[Lowprice]
            ,[ClosePrice]
            ,[Change]
            ,[Volume]
            ,[Amount]
            ,[TradedMarketValue]
            ,[MarketValue]
            ,[Turnover]
            ,[AdjPrice]
            ,[ReportType]
            ,[ReportDate]
            ,[PE_TTM]
            ,[PS_TTM]
            ,[PC_TTM]
            ,[PB]
            ,[PE_TTM_Deducted]
        FROM [FinanceData].[dbo].[AShare_PriceDaily]
    """
    ,'ahsare_indexvalue':
    """
            SELECT  [Code]
            ,[Date]
            ,[PreClose]
            ,[OpenPrice]
            ,[HighPrice]
            ,[LowPrice]
            ,[ClosePrice]
            ,[Change]
            ,[Volume]
            ,[Amount]
            ,[MarketValue]
            ,[TurnoverRate]
            ,[PE_TTM]
            ,[PB]
        FROM [FinanceData].[dbo].[AShare_IndexValue]
    """
    ,'ashare_indexmember':
    """
    SELECT  [S_IRDCODE]
        ,[CODE]
        ,[S_CON_INDATE]
        ,[S_CON_OUTDATE]
        ,[CUR_SIGN]
        ,[UPDATE_GTJA]
        ,[AddTime]
    FROM [FinanceData].[dbo].[AINDEXMEMBERS]
    """
}


In [ ]:
"""主函数 - 演示如何使用"""

db_config = {
    'server': '192.168.1.111',  # 服务器地址
    'database': 'FinanceData',  # 数据库名称
    'username': 'lyyz',  # 用户名
    'password': 'AAzz0011'  # 密码
}

# 创建数据库连接器
db_connector = SQLServerConnector(**db_config)

try:
    # 连接数据库
    if not db_connector.connect():
        logger.error("无法连接到数据库，程序退出")
    now_query='ashare_pricedaily'
    df=db_connector.execute_query(queries[now_query])
finally:
    # 关闭数据库连接
    db_connector.close()


In [ ]:
df.to_csv(f'./.qlib/mid_data/{now_query}.csv',encoding='utf-8',index=False)


In [ ]:
df.to_parquet(f'./.qlib/mid_data/{now_query}.parquet',index=False)

# 1 获取价格相关因子数据

### 1.1 构建因子并补全时间序列数据

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
factor_csv_file_path='./.qlib/mid_data/change.csv'
df_report=pd.read_csv(factor_csv_file_path)


In [ ]:
df_report=df_report.dropna(subset=['change'])
df_report=df_report.sort_values(by=['code','datetime'])
df_report['std_3m']=df_report.groupby('code')['change'].rolling(63).std().reset_index(level=0,drop=True)

In [ ]:
df_report=df_report[df_report['datetime'].notna()]
df_report['datetime']=pd.to_datetime(df_report['datetime'],format='%Y-%m-%d')
df_report=df_report[df_report['datetime']>'2010-01-01']


In [ ]:
# 创建完整时间序列
full_calendar = (
    df_report.groupby('code', group_keys=False)
      .apply(lambda g: pd.DataFrame({
          'code': g.name,
          'datetime': pd.date_range(g['datetime'].min(), g['datetime'].max(), freq='D')
      }),include_groups=False)
      .reset_index(drop=True)
)

In [ ]:
# 合并完整时间序列
result_df = (
    full_calendar
      .merge(df_report, on=['code', 'datetime'], how='left')
      .sort_values(['code', 'datetime']))

# 因子计算

### 1.2 获取风险警示数据并对信号做特殊处理

In [ ]:
df_risk_announce=pd.read_csv('./.qlib/mid_data/RiskAnnouncements.csv')
df_risk_announce['datetime'] = pd.to_datetime(df_risk_announce['datetime'],format='mixed')
df_risk_announce['datetime'] = pd.to_datetime(df_risk_announce['datetime'],format='%Y-%m-%d')
result_df_risk = result_df.merge(df_risk_announce, on=['code','datetime'], how='left')
result_df_risk['std_3m'] = np.where(pd.notna(result_df_risk['title']), -99998, result_df_risk['std_3m'])

In [ ]:
result_df_risk[result_df_risk['std_3m']==-99998]

### 1.3获取出入指数时间并对信号做特殊处理

In [ ]:
result_df_outindex=result_df_risk.copy()
df_aindexmember=pd.read_csv('./.qlib/mid_data/AINDEXMEMBERS.csv')

In [ ]:
df_aindexmember['S_CON_OUTDATE']=pd.to_datetime(df_aindexmember['S_CON_OUTDATE'])
df_aindexmember['datetime']=df_aindexmember['S_CON_OUTDATE']


In [ ]:
df_aindexmember.head(5)

In [ ]:
result_df_outindex=result_df_outindex.merge(df_aindexmember, on=['code','datetime'], how='left')  
result_df_outindex['std_3m'] = np.where(pd.notna(result_df_outindex['S_CON_OUTDATE']), -99999, result_df_outindex['std_3m'])


In [ ]:

columns_to_drop = ['title', 'S_IRDCODE', 'S_CON_INDATE', 'S_CON_OUTDATE', 'change']
result_df_outindex.drop(columns=columns_to_drop, inplace=True)


In [ ]:
result_df_outindex.info

### 1.4 保存数据

In [ ]:
result_df_outindex.to_csv(f'./.qlib/indicator_data/allAshare_std_3m.csv')

In [ ]:
# 指数字典
CODE_TO_INDEX_NAME = {
    '000016.SH': 'shangzheng50',
    '000300.SH': 'hushen300',
    '000905.SH': 'zhongzheng500',
    '000852.SH': 'zhongzheng1000',
}


In [ ]:
# 保存特定指数数据
for key,value in CODE_TO_INDEX_NAME.items():
    # 获取特定指数的股票列表
    specific_index_codes = df_aindexmember[
        df_aindexmember['S_IRDCODE'] == key
    ]['code'].unique().tolist()

    # 筛选result_df_outindex中属于特定指数的股票
    result = result_df_outindex[
        result_df_outindex['code'].isin(specific_index_codes)
    ].copy()
    result.to_csv(f'./.qlib/indicator_data/{value}_std_3m.csv')


# 2 获取财报因子相关数据

### 2.1 构建因子并补全时间序列数据

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
factor_csv_file_path='./.qlib/mid_data/abnormal_gross_margin.csv'
df_report=pd.read_csv(factor_csv_file_path)
factor_name = Path(factor_csv_file_path).stem


In [ ]:
df_report=df_report[df_report['ANN_DT'].notna()]
df_report['ANN_DT']=pd.to_datetime(df_report['ANN_DT'],format='%Y-%m-%d')
df_report=df_report[df_report['ANN_DT']>'2010-01-01']

In [ ]:
# 创建完整时间序列
full_calendar = (
    df_report.groupby('code', group_keys=False)
      .apply(lambda g: pd.DataFrame({
          'code': g.name,
          'ANN_DT': pd.date_range(g['ANN_DT'].min(), g['ANN_DT'].max(), freq='D')
      }),include_groups=False)
      .reset_index(drop=True)
)

In [ ]:
# 合并完整时间序列
result_df = (
    full_calendar
      .merge(df_report, on=['code', 'ANN_DT'], how='left')
      .sort_values(['code', 'ANN_DT']))
result_df = result_df.rename(columns={'ANN_DT': 'datetime'})

# 因子计算

In [ ]:
# 计算前三高管薪酬
result_df = (df_report.groupby(['code', 'ANN_DT'])['manager_return']
          .apply(lambda x: np.log(x.nlargest(3).sum()) if x.nlargest(3).sum() > 0 else np.nan)
          .reset_index(name='Top3_return_log'))
df_report=result_df.copy()

In [ ]:
# 异常毛利率
result_df[factor_name] = result_df.groupby('code')[factor_name].ffill()
result_df=result_df[['code','datetime',factor_name]]

In [ ]:
# ROE/ROA计算
df_report['TOT_NET_ASSETS_LY']=df_report['TOT_ASSETS_LY']-df_report['TOT_LIAB_LY']
df_report['cur_ROE']=df_report['NET_PROFIT_INCL_MIN_INT_INC']/df_report['TOT_NET_ASSETS']
df_report['last_ROE']=df_report['NET_PROFIT_INCL_MIN_INT_INC_LY']/df_report['TOT_NET_ASSETS_LY']
df_report['Delta_ROE']=df_report['cur_ROE']-df_report['last_ROE']

### 2.2 获取风险警示数据并对信号做特殊处理

In [ ]:
df_risk_announce=pd.read_csv('./.qlib/mid_data/RiskAnnouncements.csv')
df_risk_announce['datetime'] = pd.to_datetime(df_risk_announce['datetime'],format='mixed')
df_risk_announce['datetime'] = pd.to_datetime(df_risk_announce['datetime'],format='%Y-%m-%d')
result_df_risk = result_df.merge(df_risk_announce, on=['datetime','code'], how='left')
result_df_risk[factor_name] = np.where(pd.notna(result_df_risk['title']), -99998, result_df_risk[factor_name])

### 2.3获取出入指数时间并对信号做特殊处理

In [ ]:
df_aindexmember=pd.read_csv('./.qlib/mid_data/AINDEXMEMBERS.csv')

In [ ]:
df_aindexmember['S_CON_INDATE']=pd.to_datetime(df_aindexmember['S_CON_INDATE'])
df_aindexmember['S_CON_OUTDATE']=pd.to_datetime(df_aindexmember['S_CON_OUTDATE'])
df_aindexmember['datetime']=df_aindexmember['S_CON_OUTDATE']

In [ ]:
result_df_outindex=result_df_risk.copy()
result_df_outindex=result_df_outindex.merge(df_aindexmember, on=['code','datetime'], how='left')
result_df_outindex[factor_name] = np.where(pd.notna(result_df_outindex['S_CON_OUTDATE']), -99999, result_df_outindex[factor_name])
final_result=result_df_outindex.copy()

### 2.4 保存数据

In [ ]:
# 指数字典
CODE_TO_INDEX_NAME = {
    '000016.SH': 'shangzheng50',
    '000300.SH': 'hushen300',
    '000905.SH': 'zhongzheng500',
    '000852.SH': 'zhongzheng1000',
    '000001.SH': 'allAshare'
}


In [ ]:
# 保存特定指数数据
index_code='000905.SH'

specific_data = result_df_outindex[result_df_outindex['S_IRDCODE']==index_code]
specific_data.to_csv(f'./.qlib/indicator_data/{CODE_TO_INDEX_NAME[index_code]}_{factor_name}.csv')

In [ ]:
# 保存全A数据
final_result.to_csv(f'./.qlib/indicator_data/allAshare_{factor_name}.csv')

# 取股票数据：筛选停牌的行并将停牌行的数据填充NaN

In [ ]:
import numpy as np
print(df.info())
print(df.head())
print(df.tail())


In [ ]:
# 修正数据类型问题 - 涨停跌停判断
import pandas as pd

# 首先检查数据类型
print("数据类型检查:")
print(df[['open', 'close']].dtypes)
print("\n数据样本:")
print(df[['open', 'close']].head())

# 将价格列转换为数值类型
print("\n转换数据类型...")
df['open'] = pd.to_numeric(df['open'], errors='coerce')
df['high'] = pd.to_numeric(df['high'], errors='coerce')
df['low'] = pd.to_numeric(df['low'], errors='coerce')
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['volume'] = pd.to_numeric(df['volume'], errors='coerce')

print("转换后的数据类型:")
print(df[['open', 'close']].dtypes)
print(f"转换后open列空值数量: {df['open'].isnull().sum()}")
print(f"转换后close列空值数量: {df['close'].isnull().sum()}")



In [ ]:
# 去除open列为空值的行
# 修正语法：使用isnull()和notnull()方法
print("去除空值前的数据形状:", df.shape)
print("open列空值统计:", df['open'].isnull().sum())

# 方法1：使用notnull()保留非空值
df = df[df['open'].notnull()]

print("去除空值后的数据形状:", df.shape)
print("验证open列是否还有空值:", df['open'].isnull().sum())


In [ ]:
# 仅保留以下列不被置为 NaN
protected_columns = ['code', 'datetime', 'name', 'tradestatus','PB','open','high250_processed','abnormal_gross_margin_processed_ffill']

# 找出需要设置为NaN的列（除保护列外的所有列）
columns_to_modify = [col for col in df.columns if col not in protected_columns]

# 创建掩码：标记所有停牌的行
mask = df['tradestatus'] == '停牌'

# 将需要修改的列设置为NaN
df.loc[mask, columns_to_modify] = np.nan



In [ ]:
# 填充factor为1.0
df['factor']=1.0
print(df.head())
print(df.tail())



In [ ]:
df=df.drop_duplicates()

In [ ]:
print(df.info())
print(df.isnull().sum())
print(df.shape)


In [ ]:

import pandas as pd


# 确保数据按日期排序，以便计算前一天的价格
df = df.sort_values(['code', 'datetime'])

# 计算前一天的收盘价（使用shift操作）
df['prev_close'] = df.groupby('code')['close'].shift(1)

# 设置涨停和跌停条件
df['limit_up'] = False
df['limit_down'] = False

# 当开盘价大于前一天收盘价的110%时，标记为涨停
df.loc[df['open'] > df['prev_close'] * 1.10, 'limit_up'] = True

# 当开盘价小于前一天收盘价的90%时，标记为跌停
df.loc[df['open'] < df['prev_close'] * 0.90, 'limit_down'] = True

print(f"\n涨停股票数量: {df['limit_up'].sum()}")
print(f"跌停股票数量: {df['limit_down'].sum()}")

# 删除临时列（可选）
df = df.drop('prev_close', axis=1)


In [ ]:
print(df.info())
print(df.isnull().sum())
print(df.shape)

In [ ]:
# 定义目标文件夹路径
folder_path = f'./.qlib/csv_stock_data/csv_data_{now_query}'

# 核心步骤：创建文件夹
os.makedirs(folder_path, exist_ok=True)  # 使用方法一

for stock_code, group_df in df.groupby('code'):
    print(f"股票代码: {stock_code}")
    group_df.to_csv(f'./.qlib/csv_stock_data/csv_data_{now_query}/{stock_code}.csv', index=False)


# 转化为Qlib可用的数据（注意修改字段）

In [ ]:
!python scripts/dump_bin.py dump_all --data_path ".\.qlib\csv_stock_data\csv_data_{now_query}" --qlib_dir ".\.qlib\qlib_data\{now_query}" --symbol_field_name code --date_field_name datetime --freq day --include_fields open,high,low,close,volume,factor,abnormal_gross_margin_processed_ffill,limit_up,limit_down